# 🏠 Real Estate Analysis Crew

## What We're Building

A 4-agent real estate investment analysis pipeline:

| Agent | Role |
|-------|------|
| **Market Researcher** | Analyze local market conditions and trends |
| **Price Analyst** | Evaluate property value and comparable sales |
| **Neighborhood Scorer** | Score the neighborhood on livability factors |
| **Risk Reviewer** | Assess investment risks and provide recommendation |

## Key CrewAI Concept: Task Callbacks

We use a `callback` on the risk review task to log the final risk score
when the task completes.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Property Details

In [ ]:
property_listing = """
PROPERTY LISTING:
Address: 1425 Oak Street, Austin, TX 78704
Type: Single-family home
Beds/Baths: 3BR / 2BA
Sq Ft: 1,850
Year Built: 2018
List Price: $485,000
HOA: None
Lot Size: 0.18 acres
Garage: 2-car attached
Features: Updated kitchen, hardwood floors, smart home system,
          energy-efficient windows, covered patio
Neighborhood: South Congress (SoCo) area
School District: Austin ISD (rated 7/10)
Walk Score: 72
Days on Market: 14

SELLER MOTIVATION: Relocating for work. Priced $10K below recent comp.

COMPARABLE SALES (last 6 months):
- 1501 Oak St: 3BR/2BA, 1,900 sqft, sold $498,000 (45 days ago)
- 1380 Oak St: 3BR/2BA, 1,750 sqft, sold $472,000 (90 days ago)
- 1600 Congress Ave: 4BR/3BA, 2,200 sqft, sold $545,000 (30 days ago)
- 1200 Mary St: 3BR/2BA, 1,600 sqft, sold $440,000 (75 days ago)
"""

investment_goal = "Buy-and-hold rental property. Target 7%+ annual return."

print("Property listing loaded")

## Step 3 — Create Agents

In [ ]:
market_researcher = Agent(
    role="Real Estate Market Researcher",
    goal="Analyze local market conditions, trends, and economic factors",
    backstory=(
        "You are a real estate market analyst who covers the Austin, TX market. "
        "You track inventory levels, median prices, days on market, and economic "
        "indicators. You know Austin's tech-driven growth and its impact on housing."
    ),
    llm=llm,
    verbose=True,
)

price_analyst = Agent(
    role="Property Valuation Analyst",
    goal="Determine fair market value using comparable sales analysis",
    backstory=(
        "You are a certified appraiser with 20 years of experience. You use CMA "
        "(Comparative Market Analysis), price-per-sqft analysis, and adjustment "
        "methods to determine if a property is fairly priced, overpriced, or a deal."
    ),
    llm=llm,
    verbose=True,
)

neighborhood_scorer = Agent(
    role="Neighborhood Quality Assessor",
    goal="Score the neighborhood across key livability and investment factors",
    backstory=(
        "You evaluate neighborhoods like a resident would. You score areas on "
        "safety, schools, walkability, amenities, transit, appreciation trends, "
        "and rental demand. You know that location drives 80% of real estate value."
    ),
    llm=llm,
    verbose=True,
)

risk_reviewer = Agent(
    role="Investment Risk Analyst",
    goal="Assess all risk factors and provide a final investment recommendation",
    backstory=(
        "You are a real estate investment advisor who has analyzed 500+ deals. "
        "You calculate cash-on-cash return, cap rate, and cash flow projections. "
        "You are conservative — you'd rather miss a deal than recommend a bad one."
    ),
    llm=llm,
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks with Callback

In [ ]:
# Callback function — runs when the risk review task completes
def risk_review_callback(output):
    print("\n🔔 CALLBACK: Risk review task completed!")
    print(f"   Summary: {output.raw[:150]}...")


market_task = Task(
    description=f"""Analyze the local real estate market for this property:

{property_listing}

Cover:
1. Austin housing market overview (current state: buyer's/seller's market?)
2. South Congress (SoCo) submarket trends
3. Median prices, price trends (YoY), inventory levels
4. Key economic drivers (tech employers, population growth)
5. Interest rate environment and its impact
6. Rental market conditions (average rents for 3BR in the area)""",
    expected_output="A market analysis report for Austin/SoCo area.",
    agent=market_researcher,
)

price_task = Task(
    description=f"""Perform a comparative market analysis for this property:

{property_listing}

1. Price per square foot analysis vs. comparables
2. Adjustments for differences (lot size, features, age, condition)
3. Estimated fair market value range
4. Is the list price fair, above, or below market?
5. Likely final sale price (accounting for negotiations)
6. Estimated monthly rent potential""",
    expected_output="Comparative market analysis with valuation range.",
    agent=price_analyst,
)

neighborhood_task = Task(
    description=f"""Score the neighborhood for this property:

{property_listing}

Score each factor from 1-10:
1. SAFETY: Crime rates, trends
2. SCHOOLS: District quality, nearby schools
3. WALKABILITY: Walk score, nearby amenities
4. TRANSIT: Public transport access, commute times
5. AMENITIES: Restaurants, shopping, parks, entertainment
6. APPRECIATION: Historical and projected price growth
7. RENTAL DEMAND: Vacancy rates, tenant quality

Provide an overall neighborhood score (weighted average).""",
    expected_output="Neighborhood scorecard with weighted overall score.",
    agent=neighborhood_scorer,
)

risk_task = Task(
    description=f"""Provide a final investment analysis and recommendation.

Investment goal: {investment_goal}

Using the market analysis, valuation, and neighborhood score, calculate:
1. Cash flow projection (monthly): Rent - Mortgage - Insurance - Tax - Maintenance
2. Cash-on-cash return (assume 20% down, 6.5% rate, 30-year fixed)
3. Cap rate
4. Break-even occupancy rate
5. Top 5 risks (market, property-specific, regulatory, economic, tenant)
6. Risk mitigation strategies
7. FINAL VERDICT: BUY, PASS, or NEGOTIATE (with target price)""",
    expected_output="Investment analysis with financials and buy/pass recommendation.",
    agent=risk_reviewer,
    callback=risk_review_callback,  # Fires when this task completes
)

print("4 tasks created (risk task has a callback)")

## Step 5 — Run the Crew

In [ ]:
realestate_crew = Crew(
    agents=[market_researcher, price_analyst, neighborhood_scorer, risk_reviewer],
    tasks=[market_task, price_task, neighborhood_task, risk_task],
    process=Process.sequential,
    verbose=True,
)

print("Real estate crew assembled!")
result = realestate_crew.kickoff()
print("\n✅ Analysis complete!")

In [ ]:
print("🏠 FINAL INVESTMENT RECOMMENDATION:")
print("=" * 60)
print(result.raw)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Task callback** | `callback=func` fires when a task completes |
| **CMA analysis** | Comparable sales with adjustments for property differences |
| **Neighborhood scoring** | 7-factor weighted scorecard |
| **Investment math** | Cash-on-cash return, cap rate, cash flow |

## 🔧 Extensions

- **Zillow API**: Pull real listings and comps data
- **output_pydantic**: Structure the verdict as a Pydantic model
- **Portfolio analysis**: Run on multiple properties with `kickoff_for_each()`
- **Monte Carlo**: Add a simulation agent for risk modeling